# 00. Streak Analysis — Winning/Losing Momentum in MLB

**Hypothesis:** After N consecutive wins, the probability of the NEXT win increases.  
After N consecutive losses, the probability of the NEXT loss increases.

**Method:** Pure statistical analysis on 11 seasons (2010-2019, 2021).  
No ML, no API calls — just pandas + scipy.

**Markets:** Moneyline (win/loss) — if momentum exists, we can apply it to Run Line and O/U later.

In [ ]:
# Google Colab: uncomment to install dependencies
# !pip install openpyxl scipy

import sys
import os
import warnings
warnings.filterwarnings('ignore')

# If running from notebooks/ dir, add parent to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print('Ready!')

## 1. Load Data

In [ ]:
from src.data_loader import load_all_seasons, apply_data_filters, apply_betting_filters, add_derived_odds

# Load all 11 seasons (2010-2019, 2021)
games = load_all_seasons()
print(f'Loaded: {len(games)} games')

# Apply data quality filters
games = apply_data_filters(games)
print(f'After data filters: {len(games)} games')

# Add decimal odds and implied probabilities
games = add_derived_odds(games)

# For streak analysis, we use ALL games (including COL) to build accurate streaks.
# We'll filter to bettable games only when calculating ROI.
print(f'\nSeasons: {sorted(games["season"].unique())}')
print(f'Teams: {sorted(games["home_team"].unique())}')
print(f'Date range: {games["date"].min()} to {games["date"].max()}')

## 2. Build Team Game Sequences

For each team in each season, build a chronological sequence of W/L results.  
Then for each game, calculate the current streak entering that game.

In [ ]:
def build_team_games(games_df: pd.DataFrame) -> pd.DataFrame:
    """Expand game-level data to team-level (each game appears twice: once per team).
    
    Vectorized approach for speed — avoids iterrows().
    """
    # Home team DataFrame
    home = pd.DataFrame({
        'season': games_df['season'].values,
        'date': games_df['date'].values,
        'team': games_df['home_team'].values,
        'opponent': games_df['away_team'].values,
        'is_home': True,
        'won': games_df['home_win'].astype(bool).values,
        'runs_scored': games_df['home_final'].values,
        'runs_allowed': games_df['away_final'].values,
        'close_ml': games_df['home_close_ml'].values,
        'decimal_odds': games_df['home_decimal_odds'].values,
        'implied_prob': games_df['home_implied_prob'].values,
        'is_favorite': (games_df['home_implied_prob'] > 0.5).values,
        'involves_col': games_df['involves_col'].values,
        'is_september': games_df['is_september'].values,
        'is_extreme_line': games_df['is_extreme_line'].values,
    })

    # Away team DataFrame
    away = pd.DataFrame({
        'season': games_df['season'].values,
        'date': games_df['date'].values,
        'team': games_df['away_team'].values,
        'opponent': games_df['home_team'].values,
        'is_home': False,
        'won': (~games_df['home_win'].astype(bool)).values,
        'runs_scored': games_df['away_final'].values,
        'runs_allowed': games_df['home_final'].values,
        'close_ml': games_df['away_close_ml'].values,
        'decimal_odds': games_df['away_decimal_odds'].values,
        'implied_prob': games_df['away_implied_prob'].values,
        'is_favorite': (games_df['away_implied_prob'] > 0.5).values,
        'involves_col': games_df['involves_col'].values,
        'is_september': games_df['is_september'].values,
        'is_extreme_line': games_df['is_extreme_line'].values,
    })

    team_games = pd.concat([home, away], ignore_index=True)
    team_games = team_games.sort_values(['team', 'season', 'date']).reset_index(drop=True)
    return team_games


team_games = build_team_games(games)
print(f'Team-game rows: {len(team_games)}')
print(f'Unique teams: {team_games["team"].nunique()}')
team_games.head(10)

In [ ]:
def calculate_streaks(team_games: pd.DataFrame) -> pd.DataFrame:
    """Calculate current streak for each game (entering the game).
    
    W3 = team has won 3 in a row BEFORE this game.
    L5 = team has lost 5 in a row BEFORE this game.
    
    Uses groupby + vectorized cumsum for speed.
    """
    df = team_games.copy()
    
    # For each team+season group, compute streaks
    streaks = []
    streak_types = []
    
    for (team, season), group in df.groupby(['team', 'season']):
        group = group.sort_values('date')
        won = group['won'].values
        
        s_vals = np.zeros(len(group), dtype=int)
        s_types = ['none'] * len(group)
        
        current_streak = 0
        current_type = 'none'
        
        for i in range(len(group)):
            # Record the streak ENTERING this game
            s_vals[i] = current_streak
            s_types[i] = current_type
            
            # Update streak based on this game's result
            if won[i]:
                if current_type == 'W':
                    current_streak += 1
                else:
                    current_streak = 1
                    current_type = 'W'
            else:
                if current_type == 'L':
                    current_streak += 1
                else:
                    current_streak = 1
                    current_type = 'L'
        
        streaks.extend(s_vals)
        streak_types.extend(s_types)
    
    # Sort order must match — df is already sorted by team, season, date
    df['streak'] = streaks
    df['streak_type'] = streak_types
    
    # Create signed streak: positive = win streak, negative = loss streak
    df['signed_streak'] = np.where(
        df['streak_type'] == 'W', df['streak'],
        np.where(df['streak_type'] == 'L', -df['streak'], 0)
    )
    
    return df


team_games = calculate_streaks(team_games)
print('Streak distribution:')
print(team_games['signed_streak'].describe())
print(f'\nMax win streak entering game: {team_games["signed_streak"].max()}')
print(f'Max loss streak entering game: {team_games["signed_streak"].min()}')

## 3. Win Rate After Streak N

Core analysis: group by streak length, calculate next-game win rate.

In [ ]:
# Filter: only games with a meaningful streak (>= 1 game)
streaked = team_games[team_games['streak'] >= 1].copy()

# Group by signed streak
streak_stats = streaked.groupby('signed_streak').agg(
    games=('won', 'count'),
    wins=('won', 'sum'),
    win_rate=('won', 'mean'),
).reset_index()

# Add confidence intervals (95%)
streak_stats['ci_low'] = streak_stats.apply(
    lambda r: stats.binom.interval(0.95, int(r['games']), r['win_rate'])[0] / r['games']
    if r['games'] > 0 else 0,
    axis=1
)
streak_stats['ci_high'] = streak_stats.apply(
    lambda r: stats.binom.interval(0.95, int(r['games']), r['win_rate'])[1] / r['games']
    if r['games'] > 0 else 1,
    axis=1
)

# Baseline win rate (all games)
baseline_wr = team_games['won'].mean()
print(f'Baseline win rate: {baseline_wr:.4f} ({baseline_wr:.1%})')
print()

# Show table
display_stats = streak_stats[
    (streak_stats['signed_streak'] >= -10) & (streak_stats['signed_streak'] <= 10)
    & (streak_stats['games'] >= 50)  # Minimum sample size
].copy()
display_stats['win_rate_pct'] = (display_stats['win_rate'] * 100).round(1)
display_stats['vs_baseline'] = ((display_stats['win_rate'] - baseline_wr) * 100).round(1)

print('Win Rate After Streak (min 50 games):')
print('=' * 80)
for _, row in display_stats.iterrows():
    streak_label = f"W{int(row['signed_streak'])}" if row['signed_streak'] > 0 else f"L{int(abs(row['signed_streak']))}"
    arrow = '↑' if row['vs_baseline'] > 1.0 else '↓' if row['vs_baseline'] < -1.0 else '→'
    print(f"  After {streak_label:>3}: {row['win_rate_pct']:5.1f}% win rate  "
          f"({int(row['wins'])}/{int(row['games'])} games)  "
          f"{arrow} {row['vs_baseline']:+.1f}% vs baseline")

In [ ]:
# Visualization: Win Rate vs Streak Length
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Bar chart with CIs
plot_data = display_stats.sort_values('signed_streak')
colors = ['#d32f2f' if s < 0 else '#388e3c' for s in plot_data['signed_streak']]

ax = axes[0]
bars = ax.bar(plot_data['signed_streak'], plot_data['win_rate'] * 100, color=colors, alpha=0.8)
ax.axhline(y=baseline_wr * 100, color='gray', linestyle='--', linewidth=2, label=f'Baseline ({baseline_wr:.1%})')
ax.set_xlabel('Streak Entering Game (negative = losses, positive = wins)')
ax.set_ylabel('Win Rate (%)')
ax.set_title('Next-Game Win Rate After Streak')
ax.legend()
ax.set_ylim(35, 65)

# Plot 2: Win rate with error bars
ax2 = axes[1]
yerr_low = (plot_data['win_rate'] - plot_data['ci_low']) * 100
yerr_high = (plot_data['ci_high'] - plot_data['win_rate']) * 100
ax2.errorbar(
    plot_data['signed_streak'], plot_data['win_rate'] * 100,
    yerr=[yerr_low, yerr_high],
    fmt='o-', capsize=4, color='steelblue', markersize=6
)
ax2.axhline(y=baseline_wr * 100, color='gray', linestyle='--', linewidth=2, label=f'Baseline ({baseline_wr:.1%})')
ax2.set_xlabel('Streak Entering Game')
ax2.set_ylabel('Win Rate (%) with 95% CI')
ax2.set_title('Win Rate with Confidence Intervals')
ax2.legend()
ax2.set_ylim(35, 65)

plt.tight_layout()
plt.show()

## 4. Statistical Significance Tests

In [ ]:
# Binomial test: is win rate after streak N significantly different from baseline?
print('Statistical Significance (Binomial Test vs Baseline):')
print('=' * 80)

for _, row in display_stats.iterrows():
    n = int(row['games'])
    k = int(row['wins'])
    # Two-sided binomial test against baseline
    p_value = stats.binomtest(k, n, baseline_wr).pvalue
    streak_label = f"W{int(row['signed_streak'])}" if row['signed_streak'] > 0 else f"L{int(abs(row['signed_streak']))}"
    sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''
    print(f"  After {streak_label:>3}: p={p_value:.4f} {sig:>3}  "
          f"(win rate: {row['win_rate']:.3f}, n={n})")

## 5. Split Analysis: Favorites vs Underdogs on Streaks

In [ ]:
# Do streaking favorites perform differently than streaking underdogs?
for fav_label, fav_mask in [('Favorites', True), ('Underdogs', False)]:
    subset = streaked[streaked['is_favorite'] == fav_mask]
    base = subset['won'].mean()
    
    print(f'\n{fav_label} (baseline: {base:.1%}):')
    print('-' * 60)
    
    sub_stats = subset.groupby('signed_streak').agg(
        games=('won', 'count'),
        win_rate=('won', 'mean'),
    ).reset_index()
    
    sub_stats = sub_stats[
        (sub_stats['signed_streak'] >= -8) & (sub_stats['signed_streak'] <= 8)
        & (sub_stats['games'] >= 30)
    ]
    
    for _, row in sub_stats.iterrows():
        streak_label = f"W{int(row['signed_streak'])}" if row['signed_streak'] > 0 else f"L{int(abs(row['signed_streak']))}"
        diff = (row['win_rate'] - base) * 100
        print(f"  After {streak_label:>3}: {row['win_rate']:.1%} ({int(row['games'])} games) {diff:+.1f}% vs base")

## 6. Split Analysis: Home vs Away on Streaks

In [ ]:
for location, loc_mask in [('Home', True), ('Away', False)]:
    subset = streaked[streaked['is_home'] == loc_mask]
    base = subset['won'].mean()
    
    print(f'\n{location} Games (baseline: {base:.1%}):')
    print('-' * 60)
    
    sub_stats = subset.groupby('signed_streak').agg(
        games=('won', 'count'),
        win_rate=('won', 'mean'),
    ).reset_index()
    
    sub_stats = sub_stats[
        (sub_stats['signed_streak'] >= -8) & (sub_stats['signed_streak'] <= 8)
        & (sub_stats['games'] >= 30)
    ]
    
    for _, row in sub_stats.iterrows():
        streak_label = f"W{int(row['signed_streak'])}" if row['signed_streak'] > 0 else f"L{int(abs(row['signed_streak']))}"
        diff = (row['win_rate'] - base) * 100
        print(f"  After {streak_label:>3}: {row['win_rate']:.1%} ({int(row['games'])} games) {diff:+.1f}% vs base")

## 7. ROI Simulation: Betting on Streaks

If momentum exists, what would the ROI be with flat betting on closing moneyline?

In [ ]:
# Filter to bettable games only (no Colorado, no September, no extreme lines)
bettable = streaked[
    ~streaked['involves_col']
    & ~streaked['is_september']
    & ~streaked['is_extreme_line']
].copy()

print(f'Bettable games with streaks: {len(bettable)}')


def simulate_streak_betting(df: pd.DataFrame, min_streak: int, streak_type: str, 
                            unit_size: float = 100.0) -> dict:
    """Simulate flat betting on teams with a given streak.
    
    Args:
        df: Team-game DataFrame with streaks and odds
        min_streak: Minimum streak length to trigger a bet
        streak_type: 'W' (bet on winning team) or 'L' (bet against losing team)
        unit_size: Flat bet size
    
    Returns:
        Dict with ROI, win_rate, n_bets, total_pnl
    """
    if streak_type == 'W':
        # Bet on teams with N+ consecutive wins (back the streaker)
        bets = df[df['signed_streak'] >= min_streak].copy()
    elif streak_type == 'L':
        # Bet AGAINST teams with N+ consecutive losses (fade the cold team)
        # This means we bet on their OPPONENT, so we flip 'won'
        bets = df[df['signed_streak'] <= -min_streak].copy()
        bets['won'] = ~bets['won']
        # Use opponent's odds (approximate: flip implied prob)
        bets['decimal_odds'] = 1 / (1 - 1/bets['decimal_odds'])  # rough flip
    else:
        raise ValueError(f'Unknown streak_type: {streak_type}')
    
    if len(bets) == 0:
        return {'n_bets': 0, 'win_rate': 0, 'roi': 0, 'total_pnl': 0}
    
    # Calculate P&L for each bet (flat staking)
    bets['pnl'] = bets.apply(
        lambda r: unit_size * (r['decimal_odds'] - 1) if r['won'] else -unit_size,
        axis=1
    )
    
    total_staked = len(bets) * unit_size
    total_pnl = bets['pnl'].sum()
    
    return {
        'n_bets': len(bets),
        'win_rate': bets['won'].mean(),
        'roi': total_pnl / total_staked if total_staked > 0 else 0,
        'total_pnl': total_pnl,
        'avg_odds': bets['decimal_odds'].mean(),
    }


# Test different streak thresholds
print('\n=== Betting ON winning streaks (back the hot team) ===')
print(f'{"Streak":>8} {"Bets":>6} {"Win%":>7} {"Avg Odds":>9} {"ROI":>8} {"P&L":>10}')
print('-' * 55)
for min_s in range(1, 9):
    result = simulate_streak_betting(bettable, min_s, 'W')
    if result['n_bets'] >= 50:
        print(f"  W{min_s}+  {result['n_bets']:6d} {result['win_rate']:6.1%} {result['avg_odds']:9.2f} "
              f"{result['roi']:7.1%} {result['total_pnl']:10.0f}")

print('\n=== Betting AGAINST losing streaks (fade the cold team) ===')
print(f'{"Streak":>8} {"Bets":>6} {"Win%":>7} {"Avg Odds":>9} {"ROI":>8} {"P&L":>10}')
print('-' * 55)
for min_s in range(1, 9):
    result = simulate_streak_betting(bettable, min_s, 'L')
    if result['n_bets'] >= 50:
        print(f"  L{min_s}+  {result['n_bets']:6d} {result['win_rate']:6.1%} {result['avg_odds']:9.2f} "
              f"{result['roi']:7.1%} {result['total_pnl']:10.0f}")

## 8. Streak Duration Distribution

In [ ]:
# How often do long streaks occur?
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Win streaks
win_streaks = team_games[team_games['streak_type'] == 'W']['streak']
axes[0].hist(win_streaks, bins=range(1, 16), color='#388e3c', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Win Streak Length')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Win Streaks Entering Game')

# Loss streaks
loss_streaks = team_games[team_games['streak_type'] == 'L']['streak']
axes[1].hist(loss_streaks, bins=range(1, 16), color='#d32f2f', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Loss Streak Length')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Loss Streaks Entering Game')

plt.tight_layout()
plt.show()

# Stats
for s_type, label in [('W', 'Win'), ('L', 'Loss')]:
    s = team_games[team_games['streak_type'] == s_type]['streak']
    print(f'{label} streaks: mean={s.mean():.1f}, median={s.median():.0f}, '
          f'max={s.max()}, games with 5+: {(s >= 5).sum()}')

## 9. Year-by-Year Consistency Check

Does the momentum effect (if any) hold across all seasons, or is it driven by specific years?

In [ ]:
# Check win rate after W3+ and L3+ by season
print('Win rate after W3+ streak, by season:')
print('-' * 50)

yearly_data = []
for season in sorted(streaked['season'].unique()):
    s = streaked[streaked['season'] == season]
    
    w3 = s[s['signed_streak'] >= 3]
    l3 = s[s['signed_streak'] <= -3]
    
    w3_wr = w3['won'].mean() if len(w3) > 0 else None
    l3_wr = l3['won'].mean() if len(l3) > 0 else None
    
    yearly_data.append({
        'season': season,
        'w3_win_rate': w3_wr,
        'w3_n': len(w3),
        'l3_win_rate': l3_wr,
        'l3_n': len(l3),
    })
    
    if w3_wr is not None and l3_wr is not None:
        print(f"  {season}: After W3+ = {w3_wr:.1%} (n={len(w3)})  |  After L3+ = {l3_wr:.1%} (n={len(l3)})")

yearly_df = pd.DataFrame(yearly_data)

In [ ]:
# Visualize year-by-year
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(yearly_df['season'], yearly_df['w3_win_rate'] * 100, 'o-', color='#388e3c', 
        linewidth=2, markersize=8, label='Win rate after W3+ streak')
ax.plot(yearly_df['season'], yearly_df['l3_win_rate'] * 100, 's-', color='#d32f2f', 
        linewidth=2, markersize=8, label='Win rate after L3+ streak')
ax.axhline(y=50, color='gray', linestyle='--', linewidth=1.5, label='50% baseline')

ax.set_xlabel('Season')
ax.set_ylabel('Win Rate (%)')
ax.set_title('Yearly Win Rate After Streaks (W3+ and L3+)')
ax.legend()
ax.set_ylim(40, 60)

plt.tight_layout()
plt.show()

## 10. Conclusions

Run this cell to get a summary of findings.

In [ ]:
# Summary
print('=' * 80)
print('STREAK ANALYSIS — SUMMARY')
print('=' * 80)

# Overall momentum test
w3_games = streaked[streaked['signed_streak'] >= 3]
l3_games = streaked[streaked['signed_streak'] <= -3]

w3_wr = w3_games['won'].mean()
l3_wr = l3_games['won'].mean()

w3_p = stats.binomtest(int(w3_games['won'].sum()), len(w3_games), baseline_wr).pvalue
l3_p = stats.binomtest(int(l3_games['won'].sum()), len(l3_games), baseline_wr).pvalue

print(f'\nBaseline win rate: {baseline_wr:.1%}')
print(f'\nAfter W3+ streak: {w3_wr:.1%} (n={len(w3_games)}, p={w3_p:.4f})')
print(f'After L3+ streak: {l3_wr:.1%} (n={len(l3_games)}, p={l3_p:.4f})')

print(f'\n--- VERDICT ---')
if w3_p < 0.05:
    if w3_wr > baseline_wr:
        print('✅ HOT HAND EXISTS: Win rate after W3+ is significantly ABOVE baseline')
    else:
        print('🔄 REGRESSION: Win rate after W3+ is significantly BELOW baseline')
else:
    print('⚖️ NO MOMENTUM (wins): Win rate after W3+ not significantly different from baseline')

if l3_p < 0.05:
    if l3_wr < baseline_wr:
        print('✅ COLD STREAK EXISTS: Win rate after L3+ is significantly BELOW baseline')
    else:
        print('🔄 BOUNCE BACK: Win rate after L3+ is significantly ABOVE baseline')
else:
    print('⚖️ NO MOMENTUM (losses): Win rate after L3+ not significantly different from baseline')

print('\n--- BETTING IMPLICATION ---')
print('If no momentum found → streak-based strategies have NO edge.')
print('If momentum found → check ROI table above (Section 7) for profitability.')
print('Remember: even if win rate differs, odds already price this in → ROI may still be negative.')